# Tercera Entrega (IL1.3) - Arquitectura RAG (Retrieval-Augmented Generation)

Siguiendo con el desarrollo de nuestro chatbot para Invermar (iniciado en `ChatBot.ipynb` y cuyas técnicas de prompt fueron depuradas en `Segunda_Entrega.ipynb`), el siguiente paso es conectar el chatbot a una **Base de Conocimiento Real**.

En este notebook implementaremos RAG. Esto permitirá que el chatbot de Recursos Humanos no dependa de su conocimiento pre-entrenado, sino que consulte directamente los manuales, políticas y beneficios reales de Invermar antes de responder.

In [1]:
# Configuración inicial y variables de entorno
import os
from dotenv import load_dotenv

# Cargar variables de entorno (Asegúrate de apuntar al archivo correcto .env)
load_dotenv("/Users/RamonOsorio/Desktop/Ingenier-a-de-Soluciones-con-Inteligencia-Artificial/.env")

# Verificamos que las credenciales existan
token = os.environ.get("GITHUB_TOKEN")
base_url = os.environ.get("OPENAI_BASE_URL", "https://models.inference.ai.azure.com")

print("✓ Entorno configurado correctamente")

✓ Entorno configurado correctamente


### 1. Creación de la Base de Conocimiento (Documentos Simulados de Invermar)
En un entorno productivo leeríamos PDFs de la intranet. Aquí usaremos textos estructurados como ejemplo.

In [2]:
from langchain_core.documents import Document

# Extraemos algunas políticas ficticias (pero realistas) del área de RRHH de Invermar
politicas_invermar = [
    Document(
        page_content="Política de Aguinaldos Invermar: Todo trabajador de planta con más de 3 meses de antigüedad recibe un aguinaldo de Fiestas Patrias de $85.000 líquidos, más $15.000 por cada carga familiar acreditada.",
        metadata={"fuente": "Manual de Beneficios 2024", "categoria": "beneficios"}
    ),
    Document(
        page_content="Feriado Progresivo: En Invermar, los trabajadores tienen derecho a un día adicional de vacaciones por cada 3 nuevos años trabajados para su actual empleador, siempre que cuenten con 10 años de imposiciones continuas o discontinuas en cualquier sistema previsional.",
        metadata={"fuente": "Políticas de Vacaciones", "categoria": "vacaciones"}
    ),
    Document(
        page_content="Procedimiento de Finiquito: Una vez firmado el documento de renuncia o recibida la carta de despido, la empresa dispone de 10 días hábiles para tener el finiquito a disposición del trabajador en la Notaría de Puerto Montt o Calbuco, según corresponda a su planta.",
        metadata={"fuente": "Protocolos de Desvinculación", "categoria": "legal"}
    ),
    Document(
        page_content="Bono de Producción (Planta Procesos): El bono de producción mensual se calcula en base al volumen de materia prima procesada y se paga con un mes de desfase. Se liquida en la remuneración del mes siguiente.",
        metadata={"fuente": "Anexo de Remuneraciones Planta", "categoria": "remuneraciones"}
    )
]

print(f"✓ Se han cargado {len(politicas_invermar)} documentos en la memoria.")

✓ Se han cargado 4 documentos en la memoria.


### 2. Embeddings y Vector Store (FAISS)
Para buscar la información relevante a la pregunta del usuario, necesitamos convertir nuestros textos en vectores numéricos usando un modelo de embeddings y guardarlos en una base de datos vectorial (FAISS).

In [3]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# Configurar el modelo de embeddings
embeddings = OpenAIEmbeddings(
    base_url=base_url,
    api_key=token,
    model="text-embedding-3-small" # Modelo recomendado y eficiente
)

# Crear la base de datos vectorial con los documentos de Invermar
vectorstore = FAISS.from_documents(politicas_invermar, embeddings)

# Crear el "Retriever" que buscará los documentos más similares a la consulta
# search_kwargs={"k": 2} indica que traerá los 2 documentos más relevantes
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print("✓ Base de datos vectorial FAISS creada con éxito.")

✓ Base de datos vectorial FAISS creada con éxito.


### 3. Prueba del Retriever
Veamos si nuestro buscador es capaz de traer el documento correcto cuando un usuario pregunta algo.

In [4]:
consulta_usuario = "¿Cómo funciona el bono de producción para la planta?"

documentos_recuperados = retriever.invoke(consulta_usuario)

print(f"Consulta: '{consulta_usuario}'\n")
print("Documentos recuperados:")
for i, doc in enumerate(documentos_recuperados, 1):
    print(f"\n[{i}] Fuente: {doc.metadata['fuente']}")
    print(f"Contenido: {doc.page_content}")

Consulta: '¿Cómo funciona el bono de producción para la planta?'

Documentos recuperados:

[1] Fuente: Anexo de Remuneraciones Planta
Contenido: Bono de Producción (Planta Procesos): El bono de producción mensual se calcula en base al volumen de materia prima procesada y se paga con un mes de desfase. Se liquida en la remuneración del mes siguiente.

[2] Fuente: Manual de Beneficios 2024
Contenido: Política de Aguinaldos Invermar: Todo trabajador de planta con más de 3 meses de antigüedad recibe un aguinaldo de Fiestas Patrias de $85.000 líquidos, más $15.000 por cada carga familiar acreditada.


### 4. Cadena RAG (Retrieval-Augmented Generation)
Ahora conectamos el Retriever con el modelo de lenguaje (LLM). El LLM recibirá tanto la pregunta del usuario como los documentos recuperados para poder dar una respuesta precisa basada EXCLUSIVAMENTE en la política de la empresa.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Configurar el LLM
llm = ChatOpenAI(
    base_url=base_url,
    api_key=token,
    model="gpt-4o-mini",
    temperature=0.1 # Temperatura baja para que no invente (alucinaciones)
)

# 2. Definir el Prompt para RAG
# Aquí aplicamos lo aprendido en la Segunda Entrega sobre el diseño del prompt.
template_rag = """Eres el asistente de Recursos Humanos de Invermar S.A.
Tu tarea es responder la pregunta del trabajador utilizando ÚNICAMENTE el contexto proporcionado.
Si el contexto no contiene la respuesta, debes decir honestamente que no tienes esa información y derivar al área de RRHH.

Contexto oficial de Invermar:
{contexto}

Pregunta del trabajador: {pregunta}

Respuesta:"""

prompt_rag = ChatPromptTemplate.from_template(template_rag)

# 3. Función auxiliar para formatear los documentos
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 4. Construir la cadena RAG con LCEL (LangChain Expression Language)
cadena_rag = (
    # Primero, pasamos la pregunta al retriever para obtener el contexto
    {"contexto": retriever | format_docs, "pregunta": RunnablePassthrough()}
    | prompt_rag
    | llm
    | StrOutputParser()
)

print("✓ Cadena RAG configurada correctamente.")

### 5. Chatbot Invermar con RAG en Acción
Probemos el bot haciéndole preguntas. Notarás que sus respuestas están ancladas a las reglas que le pasamos en los documentos.

In [ ]:
consultas_prueba = [
    "¿De cuánto es el aguinaldo de fiestas patrias y quiénes lo reciben?",
    "¿Dónde firmo mi finiquito?",
    "¿A qué AFP debe estar afiliado un trabajador nuevo?" # Pregunta fuera del contexto
]

print("=== RESPUESTAS DEL CHATBOT INVERMAR (CON RAG) ===\n")

for pregunta in consultas_prueba:
    print(f"🧑 Trabajador: {pregunta}")
    respuesta = cadena_rag.invoke(pregunta)
    print(f"🤖 Bot Invermar: {respuesta}\n")
    print("-" * 50 + "\n")